# `threading.RLock` на реальному прикладі: `ConfigRegistry`

Цей notebook пояснює, коли `threading.RLock` може бути обґрунтованим вибором, а не просто “зручнішою версією `Lock`”.

Головний приклад: **registry конфігурації з callbacks/listeners**, де listener може синхронно звернутися назад до того самого registry через його публічний thread-safe API.


## 1. Ключова ідея

`RLock` потрібен не тому, що зі звичайним `Lock` “нічого не можна зробити”.
Часто систему справді можна перепроєктувати так, щоб обійтися звичайним `Lock`.
Але `RLock` стає обґрунтованим, коли ми **свідомо дозволяємо re-entrant public API calls**.

Тобто:

```text
метод registry.update_many()
    бере lock
    оновлює shared state
    викликає listener
        listener викликає registry.snapshot()
            snapshot() теж бере той самий lock
```

Якщо lock звичайний - буде self-deadlock.

Якщо lock re-entrant - той самий потік може повторно зайти в уже захоплений lock.


## 2. Мінімальна демонстрація проблеми зі звичайним `Lock`

У цьому прикладі `set()` тримає lock і викликає listener. Listener у відповідь викликає `get()` того самого registry.
Зі звичайним `threading.Lock` це призводить до self-deadlock.
Щоб notebook не зависав назавжди, ми запускаємо код в окремому thread і робимо `join(timeout=...)`, щоб комірка взагалі відпрацювала.


In [17]:
from __future__ import annotations

import threading
from collections.abc import Callable
from typing import Any


ConfigListener = Callable[[str, Any], None]


class ConfigRegistryWithLock:
    def __init__(self) -> None:
        self._values: dict[str, Any] = {}
        self._listeners: list[ConfigListener] = []
        self._lock = threading.Lock()

    def get(self, key: str) -> Any:
        with self._lock:
            return self._values.get(key)

    def set(self, key: str, value: Any) -> None:
        with self._lock:
            self._values[key] = value

            # Зовнішній callback викликається під lock.
            for listener in self._listeners:
                listener(key, value)

    def subscribe(self, listener: ConfigListener) -> None:
        with self._lock:
            self._listeners.append(listener)

In [18]:
registry = ConfigRegistryWithLock()


def reload_database_pool(changed_key: str, changed_value: Any) -> None:
    if changed_key != 'database_url':
        return

    # Listener синхронно заходить назад у той самий registry.
    # get() спробує взяти той самий Lock, який уже тримає set().
    pool_size = registry.get('database_pool_size')
    print(f'Recreating DB pool: url={changed_value}, pool_size={pool_size}')


registry.subscribe(reload_database_pool)
registry.set('database_pool_size', 10)

thread = threading.Thread(
    target=lambda: registry.set('database_url', 'postgresql://localhost/app'),
    daemon=True,
)

thread.start()
thread.join(timeout=1)

if thread.is_alive():
    print('Deadlock detected: thread is still alive after timeout')
else:
    print('Finished successfully')

Deadlock detected: thread is still alive after timeout


### Що сталося?

Послідовність така:

```text
thread A:
  registry.set("database_url", ...)
    acquire Lock
    update value
    call listener reload_database_pool()
      listener calls registry.get("database_pool_size")
        get() tries to acquire the same Lock
        waits forever
```

Це не deadlock між двома потоками. Це **self-deadlock**: потік заблокував сам себе.

Звичайний `Lock` не дозволяє одному й тому самому потоку повторно захопити lock, який він уже тримає.


## 3. Та сама модель, але з `RLock`

Тепер замінимо `threading.Lock()` на `threading.RLock()`.

`RLock` памʼятає, який потік володіє lockʼом, і дозволяє цьому ж потоку захопити його повторно.

Схематично:

```text
thread A:
  acquire RLock  # depth = 1
  acquire RLock  # depth = 2
  release RLock  # depth = 1
  release RLock  # depth = 0, lock fully released
```


In [19]:
class ConfigRegistryWithRLock:
    def __init__(self) -> None:
        self._values: dict[str, Any] = {}
        self._listeners: list[ConfigListener] = []
        self._lock = threading.RLock()

    def get(self, key: str) -> Any:
        with self._lock:
            return self._values.get(key)

    def set(self, key: str, value: Any) -> None:
        with self._lock:
            self._values[key] = value

            # Listener може легально зайти назад у цей самий registry.
            for listener in self._listeners:
                listener(key, value)

    def subscribe(self, listener: ConfigListener) -> None:
        with self._lock:
            self._listeners.append(listener)

In [20]:
registry = ConfigRegistryWithRLock()


def reload_database_pool(changed_key: str, changed_value: Any) -> None:
    if changed_key != 'database_url':
        return

    pool_size = registry.get('database_pool_size')
    print(f'Recreating DB pool: url={changed_value}, pool_size={pool_size}')


registry.subscribe(reload_database_pool)
registry.set('database_pool_size', 10)
registry.set('database_url', 'postgresql://localhost/app')

Recreating DB pool: url=postgresql://localhost/app, pool_size=10


## 4. Чому це реалістичний кейс

У реальній системі registry конфігурації може мати такий контракт:

> Коли config змінюється, listeners викликаються синхронно під час update. Listener має право звернутися до registry через public API, щоб прочитати повний актуальний стан або зареєструвати додатковий listener.

Наприклад:

```python
def listener(changes):
    full_config = registry.snapshot()
    db_url = registry.get("database_url")
    registry.subscribe(another_listener)
```

Тут важливо, що listener - це **зовнішній код**. Registry не контролює, які саме public methods callback викличе.

Саме тому варіант з приватними `_get_unlocked()` методами тут не дуже допомагає: зовнішній callback не має і не повинен викликати приватний API.


## 5. Сильніший приклад: batch update + snapshot

Тепер зробимо registry, який оновлює одразу кілька значень.
Listener хоче прочитати повний snapshot конфігу після batch update.
З `RLock` це працює, бо `update_many()` може тримати lock, а listener може синхронно викликати `snapshot()`.


In [21]:
BatchConfigListener = Callable[[dict[str, Any]], None]


class BatchConfigRegistryWithRLock:
    def __init__(self) -> None:
        self._values: dict[str, Any] = {}
        self._listeners: list[BatchConfigListener] = []
        self._lock = threading.RLock()

    def get(self, key: str) -> Any:
        with self._lock:
            return self._values.get(key)

    def snapshot(self) -> dict[str, Any]:
        with self._lock:
            return dict(self._values)

    def subscribe(self, listener: BatchConfigListener) -> None:
        with self._lock:
            self._listeners.append(listener)

    def update_many(self, values: dict[str, Any]) -> None:
        with self._lock:
            self._values.update(values)

            for listener in self._listeners:
                listener(values)

In [22]:
registry = BatchConfigRegistryWithRLock()


def rebuild_http_client(changes: dict[str, Any]) -> None:
    if 'api_base_url' not in changes:
        return

    # Listener читає повний стан registry через public API.
    config = registry.snapshot()

    timeout = config['api_timeout']
    retries = config['api_retries']
    base_url = config['api_base_url']

    print(f'Rebuild HTTP client: base_url={base_url}, timeout={timeout}, retries={retries}')


registry.subscribe(rebuild_http_client)

registry.update_many(
    {
        'api_base_url': 'https://api.example.com',
        'api_timeout': 5,
        'api_retries': 3,
    }
)

Rebuild HTTP client: base_url=https://api.example.com, timeout=5, retries=3


## 6. Чи можна прибрати `RLock` і використати звичайний `Lock`?

Так, можна.

Але зазвичай це означає не просто “краще розставити lockʼи”, а **змінити контракт системи**.

Найпоширеніший варіант: не викликати зовнішній callback під lock. Замість цього:

1. під lock оновити state;
2. під lock зробити копію listeners;
3. під lock зробити snapshot;
4. відпустити lock;
5. викликати listeners уже без lock, передавши їм snapshot.

Це часто навіть кращий дизайн, бо зовнішній код не виконується під lock.


In [23]:
SnapshotConfigListener = Callable[[dict[str, Any], dict[str, Any]], None]


class BatchConfigRegistryWithLockAndSnapshot:
    def __init__(self) -> None:
        self._values: dict[str, Any] = {}
        self._listeners: list[SnapshotConfigListener] = []
        self._lock = threading.Lock()

    def get(self, key: str) -> Any:
        with self._lock:
            return self._values.get(key)

    def snapshot(self) -> dict[str, Any]:
        with self._lock:
            return dict(self._values)

    def subscribe(self, listener: SnapshotConfigListener) -> None:
        with self._lock:
            self._listeners.append(listener)

    def update_many(self, values: dict[str, Any]) -> None:
        with self._lock:
            self._values.update(values)
            listeners = list(self._listeners)
            event_snapshot = dict(self._values)

        # Зовнішній код викликається вже без lock.
        for listener in listeners:
            listener(values, event_snapshot)

In [24]:
registry = BatchConfigRegistryWithLockAndSnapshot()


def rebuild_http_client_from_snapshot(
    changes: dict[str, Any],
    snapshot: dict[str, Any],
) -> None:
    if 'api_base_url' not in changes:
        return

    timeout = snapshot['api_timeout']
    retries = snapshot['api_retries']
    base_url = snapshot['api_base_url']

    print(f'Rebuild HTTP client: base_url={base_url}, timeout={timeout}, retries={retries}')


registry.subscribe(rebuild_http_client_from_snapshot)

registry.update_many(
    {
        'api_base_url': 'https://api.example.com',
        'api_timeout': 5,
        'api_retries': 3,
    }
)

Rebuild HTTP client: base_url=https://api.example.com, timeout=5, retries=3


## 7. Але що змінилося в дизайні?

Варіант з `RLock` має такий контракт:

```text
Listener може синхронно викликати public methods того самого registry.
```

Наприклад:

```python
def listener(changes):
    config = registry.snapshot()
```

Варіант зі звичайним `Lock` і snapshot має **інший контракт**:

```text
Listener не повинен синхронно лізти назад у registry.
Усе, що йому потрібно для реакції на подію, передається в event payload.
```

Наприклад:

```python
def listener(changes, snapshot):
    ...
```

Обидва варіанти можуть бути правильними. Але це **різні API-гарантії**. Тому, якщо необхідно зробити контракт "Listener може синхронно викликати public methods того самого registry" має бути застосований саме RLock.


## 8. Чому не завжди достатньо “правильно поставити звичайні lockʼи”?

Можна спробувати зробити два lockʼи:

```text
_values_lock
_listeners_lock
```

Але якщо `update_many()` тримає `_values_lock`, а listener викликає `snapshot()`, який теж хоче `_values_lock`, проблема лишається:

```text
update_many()
  acquire _values_lock
  call listener
    listener calls snapshot()
      snapshot tries to acquire _values_lock
      self-deadlock
```

Тобто кількість lockʼів сама по собі не вирішує проблему re-entrant входу в той самий protected state.

Щоб прибрати `RLock`, треба або:

- не викликати callbacks під lock;
- заборонити callbacks звертатися назад у registry;
- передавати snapshot/event payload;
- перейти на immutable/copy-on-write модель;
- або змінити архітектуру синхронізації.


## 9. Коли `RLock` тут справді обґрунтований

`RLock` має сенс, якщо ми свідомо хочемо підтримати такий контракт:

```text
Під час синхронного callbackʼа listener може безпечно викликати public API того самого registry.
```

Це може бути корисно, якщо:

- registry є частиною plugin/event system;
- callbacks пишуться зовнішнім кодом;
- ми хочемо дозволити listeners читати актуальний стан через public API;
- listener може викликати `get()`, `snapshot()`, `subscribe()`;
- callback має виконатися синхронно як частина update operation;
- зміна на snapshot-based API була б breaking change.

У такому випадку `RLock` - це не “ми не змогли написати з Lock”.

Це означає:

> Ми явно підтримуємо re-entrant contract для public API цього обʼєкта.


## 10. Висновок

Для нового коду часто краще починати з такого правила:

```text
Не викликати зовнішній код під lock.
```

Тобто краще:

- зробити update під lock;
- підготувати immutable snapshot або event payload;
- відпустити lock;
- викликати listeners.

Але якщо система вже має або свідомо хоче мати re-entrant callbacks, тоді `RLock` є нормальним інструментом.

Коротко:

```text
Lock  - коли один потік не має повторно заходити в той самий locked section.
RLock - коли той самий потік легально може повторно зайти в protected API.
```

Тобто:

> `RLock` майже завжди можна архітектурно прибрати, але не завжди без зміни API-контракту та гарантій системи.
